# Lab 0 — One-time setup (run this first)

Run this notebook **once per JupyterLab space**, before any lab. It does two things:

1. **Installs the shared kernel dependencies** from the repo-root `requirements.txt` (the single source of truth for packages — individual labs contain no `pip install` cells).
2. **Discovers your AWS environment once and writes a single repo-root `.env`** (region, account, project name, data bucket, execution role, MLflow tracking URI). Every lab loads *this* file — no lab discovers or hardcodes these values itself.

> **Single source of truth for the project name:** `PROJECT_NAME` is **discovered from the deployed stack** (the root stack's `ProjectName` parameter, falling back to the `<project>-monitoring-<account>` S3 Table Bucket). This matters because Workshop Studio events deploy with their own `ProjectName`, so a hardcoded default would point every lab at resources that do not exist. To force a specific name, `export PROJECT_NAME=...` before running this notebook and it flows into `.env` for every lab.

## 1. Install the shared kernel dependencies

In [ ]:
# Bootstrap the installers: keep pip current and install uv
# (fast resolver/installer used throughout these labs).
!pip install -q -U pip uv

In [ ]:
# Install the full shared kernel stack from the repo-root requirements.txt
# using uv. We remove any pre-existing SageMaker SDK packages first for a
# clean install. (requirements.txt lives at the repo root, one level up.)
import sys, os
_REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
_REQS = os.path.join(_REPO_ROOT, 'requirements.txt')
!uv pip uninstall --python {sys.executable} sagemaker sagemaker-core sagemaker-train sagemaker-serve sagemaker-mlops || true
!uv pip install --python {sys.executable} --no-cache -r {_REQS}

## 2. Restart the kernel to pick up the updated packages

In [ ]:
# Restart kernel to pick up the updated packages
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

## 3. Discover the AWS environment and write the shared `.env`

This is the **only** place environment values are discovered. It writes `<repo-root>/.env`; every lab loads that same file via the shared snippet below.

In [ ]:
# Discover AWS environment values ONCE and persist them to a single
# repo-root .env. Every lab reads this file — nothing is discovered or
# hardcoded per-lab. Re-run this cell (or delete the .env) to refresh.
import os
from pathlib import Path
import boto3

REPO_ROOT = Path.cwd().parent            # lab0-setup/ -> repo root
ENV_FILE = REPO_ROOT / '.env'

# The name this workshop uses when it deploys the stack itself. It is only a
# last-resort fallback: never assume it, because Workshop Studio events deploy
# with their own ProjectName (e.g. smai-genai-ml-std) and guessing this one
# silently points every lab at resources that do not exist.
CANONICAL_PROJECT_NAME = 'bank-marketing-prediction'

region = (os.environ.get('AWS_DEFAULT_REGION')
          or boto3.session.Session().region_name
          or 'us-east-1')
account_id = boto3.client('sts', region_name=region).get_caller_identity()['Account']
sm = boto3.client('sagemaker', region_name=region)


def _discover_project_name():
    """Return (project_name, data_bucket) from deployed resources, else ('', '').

    CloudFormation is authoritative: the workshop root stack carries ProjectName
    as a parameter, so the deployed value is read rather than guessed.
    """
    try:
        cfn = boto3.client('cloudformation', region_name=region)
        best = None
        for page in cfn.get_paginator('describe_stacks').paginate():
            for stack in page['Stacks']:
                if stack.get('RootId'):
                    continue          # nested stack — the root owns the parameter
                params = {p['ParameterKey']: p['ParameterValue']
                          for p in stack.get('Parameters') or []}
                if 'ProjectName' not in params:
                    continue
                outputs = {o['OutputKey']: o['OutputValue']
                           for o in stack.get('Outputs') or []}
                # Prefer the stack that owns the data-prep resources Lab 2 needs.
                score = (('DataPrepTableBucketName' in outputs) * 2
                         + ('DataBucketName' in outputs))
                cand = (score, stack['CreationTime'],
                        params['ProjectName'], outputs.get('DataBucketName', ''))
                if best is None or cand[:2] > best[:2]:
                    best = cand
        if best:
            return best[2], best[3]
    except Exception as e:
        print(f'  (CloudFormation discovery unavailable: {e})')

    # Fallback: 5-data-prep.yaml names its S3 Table Bucket
    # <project>-monitoring-<account_id>, so the prefix is the project name.
    try:
        s3t = boto3.client('s3tables', region_name=region)
        suffix = f'-monitoring-{account_id}'
        for tb in s3t.list_table_buckets().get('tableBuckets', []):
            if tb['name'].endswith(suffix):
                return tb['name'][:-len(suffix)], ''
    except Exception as e:
        print(f'  (S3 Tables discovery unavailable: {e})')

    return '', ''


_discovered_project, _discovered_data_bucket = _discover_project_name()
_explicit_project = os.environ.get('PROJECT_NAME', '')

# Precedence: explicit override, then discovery, then the canonical fallback.
# An env var equal to CANONICAL_PROJECT_NAME is NOT treated as an override —
# that is usually a stale .env already loaded into this kernel, and honouring it
# would defeat discovery.
if _explicit_project and _explicit_project != CANONICAL_PROJECT_NAME:
    PROJECT_NAME, _project_source = _explicit_project, 'PROJECT_NAME env var'
elif _discovered_project:
    PROJECT_NAME, _project_source = _discovered_project, 'discovered from deployed stack'
else:
    PROJECT_NAME, _project_source = CANONICAL_PROJECT_NAME, 'canonical fallback (nothing deployed?)'

# SageMaker execution role from the domain (best-effort).
exec_role = ''
try:
    domains = sm.list_domains()['Domains']
    if domains:
        d = sm.describe_domain(DomainId=domains[0]['DomainId'])
        exec_role = d.get('DefaultUserSettings', {}).get('ExecutionRole', '')
except Exception:
    pass

# MLflow tracking URI — most recent app matching the project name (best-effort).
mlflow_uri = ''
try:
    apps = sm.list_mlflow_apps().get('Summaries', [])
    matching = [a for a in apps if PROJECT_NAME in a.get('Name', '')] or apps
    if matching:
        matching.sort(key=lambda a: a.get('CreationTime', ''), reverse=True)
        mlflow_uri = matching[0]['Arn']
except Exception:
    pass

# CFN convention (3-sagemaker.yaml): ${ProjectName}-data-${AWS::AccountId}-${AWS::Region}
# Prefer the stack's own DataBucketName output when it belongs to this project.
if _discovered_data_bucket and PROJECT_NAME == _discovered_project:
    data_bucket = _discovered_data_bucket
else:
    data_bucket = f'{PROJECT_NAME}-data-{account_id}-{region}'

# Workshop convention: S3 prefix where lab2 writes prepared train/test data
# and lab3 reads it back. Not a deployed resource — a fixed path segment,
# overridable via the DATA_PREFIX env var.
data_prefix = os.environ.get('DATA_PREFIX', 'bank-marketing-lab')

# Iceberg schema identifiers (Glue database, S3 Tables namespace, table names).
# These are FIXED identifiers created by 5-data-prep.yaml, not project-scoped.
# Discover them from the live resources so no notebook re-declares them; fall
# back to the canonical defaults when the data-prep stack isn't deployed yet.
athena_database = 'bank_marketing'
s3t_namespace = 'bank_marketing'
training_table = 'training_data'
evaluation_table = 'evaluation_data'
try:
    table_bucket_arn = f'arn:aws:s3tables:{region}:{account_id}:bucket/{PROJECT_NAME}-monitoring-{account_id}'
    s3t = boto3.client('s3tables', region_name=region)
    _nss = s3t.list_namespaces(tableBucketARN=table_bucket_arn).get('namespaces', [])
    if _nss:
        s3t_namespace = _nss[0]['namespace'][0]
        _tbls = [t['name'] for t in
                 s3t.list_tables(tableBucketARN=table_bucket_arn, namespace=s3t_namespace).get('tables', [])]
        if 'training_data' in _tbls:
            training_table = 'training_data'
        if 'evaluation_data' in _tbls:
            evaluation_table = 'evaluation_data'
        # Glue database mirrors the S3 Tables namespace for these labs.
        athena_database = s3t_namespace
except Exception:
    pass  # data-prep stack not deployed yet — canonical fallbacks stand.

ENV_FILE.write_text(
    f'AWS_DEFAULT_REGION={region}\n'
    f'PROJECT_NAME={PROJECT_NAME}\n'
    f'ACCOUNT_ID={account_id}\n'
    f'DATA_S3_BUCKET={data_bucket}\n'
    f'SAGEMAKER_EXEC_ROLE={exec_role}\n'
    f'MLFLOW_TRACKING_URI={mlflow_uri}\n'
    f'ALERT_EMAIL=nobody@example.com\n'
    f'ATHENA_DATABASE={athena_database}\n'
    f'S3T_NAMESPACE={s3t_namespace}\n'
    f'ATHENA_TRAINING_TABLE={training_table}\n'
    f'ATHENA_EVALUATION_TABLE={evaluation_table}\n'
    f'DATA_PREFIX={data_prefix}\n'
)
print(f'\u2713 Wrote {ENV_FILE}')
print(f'  PROJECT_NAME: {PROJECT_NAME}  ({_project_source})')
print(f'  Region:       {region}')
print(f'  Account:      {account_id}')
print(f'  Data bucket:  {data_bucket}')
print(f'  MLflow URI:   {mlflow_uri or "(not found — edit .env if a lab needs it)"}')
print(f'  Exec role:    {exec_role or "(not found — edit .env if a lab needs it)"}')
print(f'  Athena DB:    {athena_database}  (namespace {s3t_namespace})')
print(f'  Tables:       {training_table}, {evaluation_table}')

## 4. How each lab consumes this

Every lab's first cell uses the **one shared loader** at the repo root (`workshop_env.py`) — no lab re-implements the loader or hardcodes any value. It finds the repo-root `.env` and returns the resolved variables:

```python
import os, sys
# Find the repo root (holds workshop_env.py) and import the shared loader.
from pathlib import Path
for _c in [os.getcwd(), *[str(p) for p in Path(os.getcwd()).parents]]:
    if os.path.exists(os.path.join(_c, 'workshop_env.py')):
        sys.path.insert(0, _c); break
from workshop_env import load_workshop_env
env = load_workshop_env()
PROJECT_NAME     = env['PROJECT_NAME']
ATHENA_DATABASE  = env['ATHENA_DATABASE']
S3T_NAMESPACE    = env['S3T_NAMESPACE']
TRAINING_TABLE   = env['TRAINING_TABLE']
EVALUATION_TABLE = env['EVALUATION_TABLE']
DATA_PREFIX      = env['DATA_PREFIX']
```

lab5's `src/config/config.py` reads the **same** `.env` keys, so every lab shares one source. Define values once here; never re-declare them.


## 5. Verify imports

If these imports work, setup succeeded. Otherwise restart the kernel and re-run section 1.

In [ ]:
import sagemaker
import sagemaker.core
import sagemaker.train
import sagemaker.serve
import sagemaker.mlops

import mlflow
import sagemaker_mlflow
import datasets

from importlib.metadata import version
for _p in ['sagemaker', 'mlflow', 'sagemaker-mlflow']:
    print(f'{_p}: {version(_p)}')